In [1]:
import os
import sqlite3
import pandas as pd
import yfinance as yf
from datetime import datetime, timedelta
from google.colab import drive

# 1. Mount Google Drive to save the database persistently
print("Connecting to Google Drive...")
drive.mount('/content/drive')

# 2. Set up the directory and database path
data_folder = '/content/drive/MyDrive/Colab Notebooks/data/'
os.makedirs(data_folder, exist_ok=True)
db_path = os.path.join(data_folder, 'commodity_prices.db')

# 3. Define the commodities and their Yahoo Finance ticker symbols
commodities = {
    'WTI_Crude_Oil': 'CL=F',
    'Aluminum': 'ALI=F',
    'Copper': 'HG=F'
}

# Connect to the SQLite database
conn = sqlite3.connect(db_path)

print("\n--- Starting Data Collection ---")

for name, ticker in commodities.items():
    print(f"\nProcessing: {name} ({ticker})")

    try:
        # Check if the commodity already has a table in the database
        query = f"SELECT name FROM sqlite_master WHERE type='table' AND name='{name}'"
        table_exists = not pd.read_sql(query, conn).empty

        if not table_exists:
            # ==========================================
            # FIRST RUN: Download 2 years of history
            # ==========================================
            print(f"  -> First run detected. Downloading 2 years of historical data...")
            data = yf.Ticker(ticker).history(period='2y')

            if data.empty:
                print(f"  -> ERROR: Failed to fetch data. Skipping {name}...")
                continue

            # Remove timezone information from dates (SQLite prefers naive datetimes)
            data.index = data.index.tz_localize(None)

            # Save the data to the database
            data.to_sql(name, conn, if_exists='replace', index=True, index_label='Date')

            # Evaluate: First Run Summary
            print(f"  -> SUCCESS: Database table created for {name}.")
            print(f"  -> Stored {len(data)} daily price records.")
            print(f"  -> Date range: {data.index.min().date()} to {data.index.max().date()}")

        else:
            # ==========================================
            # SUBSEQUENT RUNS: Append new data only
            # ==========================================
            print(f"  -> Existing database found. Checking for new data...")

            # Find the most recent date stored in the database
            last_record_query = f"SELECT Date, Close FROM {name} ORDER BY Date DESC LIMIT 1"
            last_record = pd.read_sql(last_record_query, conn)

            last_db_date_str = last_record['Date'].iloc[0]
            last_db_price = last_record['Close'].iloc[0]
            last_db_date = pd.to_datetime(last_db_date_str)

            # Fetch data from Yahoo Finance starting from the day before the last recorded date
            # (To ensure we don't miss anything due to timezone/trading hour weirdness)
            start_date = (last_db_date - timedelta(days=1)).strftime('%Y-%m-%d')
            new_data = yf.Ticker(ticker).history(start=start_date)

            if not new_data.empty:
                new_data.index = new_data.index.tz_localize(None)

                # Filter strictly for dates newer than what we already have
                new_data = new_data[new_data.index > last_db_date]

                if not new_data.empty:
                    new_data.to_sql(name, conn, if_exists='append', index=True, index_label='Date')
                    print(f"  -> SUCCESS: Added {len(new_data)} new records.")

                    # Update our 'last known' variables for the validation step below
                    last_db_price = new_data['Close'].iloc[-1]
                    last_db_date_str = new_data.index.max().strftime('%Y-%m-%d')
                else:
                    print(f"  -> Up to date. No new trading days to add since {last_db_date.date()}.")

            # ==========================================
            # EVALUATE: Validation Check
            # ==========================================
            print(f"  -> Validating data accuracy...")
            current_yf_data = yf.Ticker(ticker).history(period='1d')

            if not current_yf_data.empty:
                current_yf_price = current_yf_data['Close'].iloc[-1]

                # Round to 2 decimal places to handle minor floating-point differences
                if round(last_db_price, 2) == round(current_yf_price, 2):
                    print(f"  -> VALIDATION PASSED: Database price ({last_db_price:.2f}) matches Yahoo Finance.")
                else:
                    print(f"  -> WARNING: Price mismatch detected!")
                    print(f"     Stored DB Price (Date: {last_db_date_str[:10]}): {last_db_price:.4f}")
                    print(f"     Current Yahoo Finance Price: {current_yf_price:.4f}")
                    print(f"     *Note: If the market is currently open, live intraday prices may cause this mismatch. If the market is closed, please share this warning with AI to investigate further.*")
            else:
                print(f"  -> WARNING: Could not fetch a validation price from Yahoo Finance right now.")

    except Exception as e:
        # INTERACT: Error handling allows the script to continue even if one commodity fails
        print(f"  -> UNEXPECTED ERROR while processing {name}: {e}")
        print(f"  -> Skipping to the next commodity...")

# Close the database connection
conn.close()
print("\n--- All Data Collection Tasks Completed ---")

Connecting to Google Drive...
Mounted at /content/drive

--- Starting Data Collection ---

Processing: WTI_Crude_Oil (CL=F)
  -> First run detected. Downloading 2 years of historical data...
  -> SUCCESS: Database table created for WTI_Crude_Oil.
  -> Stored 505 daily price records.
  -> Date range: 2024-04-01 to 2026-04-01

Processing: Aluminum (ALI=F)
  -> First run detected. Downloading 2 years of historical data...
  -> SUCCESS: Database table created for Aluminum.
  -> Stored 504 daily price records.
  -> Date range: 2024-04-01 to 2026-04-01

Processing: Copper (HG=F)
  -> First run detected. Downloading 2 years of historical data...
  -> SUCCESS: Database table created for Copper.
  -> Stored 505 daily price records.
  -> Date range: 2024-04-01 to 2026-04-01

--- All Data Collection Tasks Completed ---
